In [6]:
# Install if needed
# !pip install nba_api pandas

import time

import pandas as pd

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


# Rate limiting helper
def rate_limit(seconds=0.6):
    """NBA API rate limits aggressively. Always sleep between calls."""
    time.sleep(seconds)


print("Setup complete")

Setup complete


In [7]:
# Test IDs we'll use throughout
# Using well-known, stable IDs to minimize weird edge cases

TEST_PLAYER_ID = 201566  # Russell Westbrook (long career, lots of data)
TEST_TEAM_ID = 1610612744  # Golden State Warriors
TEST_GAME_ID = "0022300001"  # First game of 2023-24 season
TEST_SEASON = "2023-24"

print(f"Test Player ID: {TEST_PLAYER_ID}")
print(f"Test Team ID: {TEST_TEAM_ID}")
print(f"Test Game ID: {TEST_GAME_ID}")
print(f"Test Season: {TEST_SEASON}")

Test Player ID: 201566
Test Team ID: 1610612744
Test Game ID: 0022300001
Test Season: 2023-24


In [8]:
## 2.3 BoxScoreAdvancedV3 - Per-game advanced stats

In [10]:
from nba_api.stats.endpoints import boxscoreadvancedv3

box_adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=TEST_GAME_ID)
rate_limit()

dfs = box_adv.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")

# DataFrame 0: Player stats
box_adv_df = dfs[0]
print(f"\nPlayerStats Shape: {box_adv_df.shape}")
print(f"\nColumns: {list(box_adv_df.columns)}")

# V3 uses camelCase, not UPPER_SNAKE_CASE
# Adjust column names accordingly
v3_display_cols = [
    "firstName",
    "familyName",
    "teamTricode",
    "minutes",
    "offensiveRating",
    "defensiveRating",
    "usagePercentage",
    "trueShootingPercentage",
    "assistPercentage",
]

# Filter to columns that exist
available_cols = [c for c in v3_display_cols if c in box_adv_df.columns]
missing_cols = [c for c in v3_display_cols if c not in box_adv_df.columns]

if missing_cols:
    print(f"\n⚠️  Missing expected columns: {missing_cols}")

print("\n--- First 5 players ---")
display(box_adv_df[available_cols].head())

# Check DataFrame 1 if it exists (usually team-level stats)
if len(dfs) > 1:
    print("\n--- DataFrame 1 (Team Stats) ---")
    print(f"Shape: {dfs[1].shape}")
    print(f"Columns: {list(dfs[1].columns)}")
box_adv_df

Number of dataframes: 2

PlayerStats Shape: (28, 37)

Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']

--- First 5 players ---


,firstName,familyName,teamTricode,minutes,offensiveRating,defensiveRating,usagePercentage,trueShootingPercentage,assistPercentage
0,Max,Strus,CLE,28:17,119.7,116.7,0.164,0.506,0.000
1,Evan,Mobley,CLE,35:51,124.7,110.4,0.181,0.538,0.167
2,Jarrett,Allen,CLE,21:07,104.3,106.4,0.154,0.683,0.000
3,Donovan,Mitchell,CLE,36:39,119.5,117.9,0.337,0.748,0.391
4,Darius,Garland,CLE,31:59,94.4,114.7,0.213,0.549,0.316



--- DataFrame 1 (Team Stats) ---
Shape: (2, 30)
Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'estimatedTeamTurnoverPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,estimatedOffensiveRating,offensiveRating,estimatedDefensiveRating,defensiveRating,estimatedNetRating,netRating,assistPercentage,assistToTurnover,assistRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1629622,Max,Strus,M. Strus,max-strus,F,,,28:17,119.7,119.7,116.7,116.7,3.0,3.0,0.000,0.00,0.0,0.000,0.036,0.018,9.1,0.500,0.506,0.164,0.164,102.68,102.68,85.56,61.0,0.026
1,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1630596,Evan,Mobley,E. Mobley,evan-mobley,F,,,35:51,124.7,124.7,110.4,110.4,14.3,14.3,0.167,2.50,25.0,0.071,0.222,0.156,10.0,0.538,0.538,0.181,0.181,103.10,103.10,85.91,77.0,0.133
2,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1628386,Jarrett,Allen,J. Allen,jarrett-allen,C,,,21:07,104.3,104.3,106.4,106.4,-2.1,-2.1,0.000,0.00,0.0,0.091,0.263,0.171,12.5,0.667,0.683,0.154,0.154,106.84,106.84,89.03,47.0,0.124
3,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1628378,Donovan,Mitchell,D. Mitchell,donovan-mitchell,G,,,36:39,119.5,119.5,117.9,117.9,1.5,1.5,0.391,3.00,24.3,0.031,0.108,0.072,8.1,0.714,0.748,0.337,0.337,101.50,101.50,84.58,77.0,0.224
4,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1629636,Darius,Garland,D. Garland,darius-garland,G,,,31:59,94.4,94.4,114.7,114.7,-20.3,-20.3,0.316,1.50,26.1,0.000,0.000,0.000,17.4,0.455,0.549,0.213,0.213,105.04,105.04,87.53,72.0,0.094
5,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1627777,Georges,Niang,G. Niang,georges-niang,,,,27:08,125.4,125.4,110.2,110.2,15.3,15.3,0.042,0.00,9.1,0.000,0.200,0.114,0.0,0.556,0.607,0.164,0.164,104.40,104.40,87.00,59.0,0.076
6,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1630171,Isaac,Okoro,I. Okoro,isaac-okoro,,,,23:01,96.0,96.0,131.4,131.4,-35.4,-35.4,0.125,2.00,33.3,0.000,0.056,0.026,16.7,0.833,0.833,0.077,0.077,105.32,105.32,87.77,50.0,0.037
7,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1627747,Caris,LeVert,C. LeVert,caris-levert,,,,30:24,111.6,111.6,114.7,114.7,-3.1,-3.1,0.167,4.00,21.1,0.000,0.200,0.098,5.3,0.417,0.472,0.205,0.205,108.15,108.15,90.12,69.0,0.075
8,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1629731,Dean,Wade,D. Wade,dean-wade,,,,2:46,12.5,12.5,125.0,125.0,-112.5,-112.5,0.000,0.00,0.0,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,138.80,138.80,115.66,8.0,-0.100
9,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,202684,Tristan,Thompson,T. Thompson,tristan-thompson,,,,2:48,28.6,28.6,166.7,166.7,-138.1,-138.1,0.000,0.00,0.0,0.000,0.500,0.143,0.0,0.000,0.000,0.000,0.000,111.43,111.43,92.86,7.0,0.000


In [11]:
# =============================================================================
# BoxScoreAdvancedV3 - FINAL SCHEMA REFERENCE
# =============================================================================

# DataFrame 0: PlayerStats (Advanced)
PLAYER_ADVANCED_V3 = {
    "identifiers": {
        "gameId": "Game identifier",
        "teamId": "Team identifier",
        "personId": "Player identifier",
    },
    "player_meta": {
        "firstName": "Player first name",
        "familyName": "Player last name",
        "nameI": "Abbreviated name",
        "position": "Position IF starter, empty if bench",
        "comment": "DNP reason / injury status",
        "jerseyNum": "Jersey number",
    },
    "ratings": {
        "offensiveRating": "Points produced per 100 possessions",
        "defensiveRating": "Points allowed per 100 possessions",
        "netRating": "ORTG - DRTG",
        "estimatedOffensiveRating": "NBA estimated ORTG",  # Slightly different calc
        "estimatedDefensiveRating": "NBA estimated DRTG",
        "estimatedNetRating": "NBA estimated net rating",
    },
    "percentages": {
        "usagePercentage": "USG% - % of team plays used while on floor",
        "trueShootingPercentage": "TS% - scoring efficiency",
        "effectiveFieldGoalPercentage": "eFG% - FG% adjusted for 3PT value",
        "assistPercentage": "AST% - % of teammate FGs assisted",
        "offensiveReboundPercentage": "ORB% - % of available offensive rebounds grabbed",
        "defensiveReboundPercentage": "DRB% - % of available defensive rebounds grabbed",
        "reboundPercentage": "TRB% - total rebound percentage",
    },
    "other": {
        "assistToTurnover": "AST/TO ratio",
        "assistRatio": "AST ratio (per 100 possessions)",
        "turnoverRatio": "TOV ratio (turnovers per 100 possessions)",
        "pace": "Possessions per 48 minutes",
        "possessions": "Total possessions played",
        "PIE": "Player Impact Estimate (0-1 scale, ~.100 is average)",
    },
    "estimated_variants": {
        "estimatedUsagePercentage": "NBA estimated USG%",
        "estimatedPace": "NBA estimated pace",
    },
}

# DataFrame 1: TeamStats (Advanced) - same columns minus player-specific ones
TEAM_ADVANCED_V3 = {
    "rows": "2 (one per team)",
    "use_case": "Game-level team efficiency, pace for both teams",
    "key_fields": ["offensiveRating", "defensiveRating", "pace", "possessions"],
}


# =============================================================================
# IMPORTANT: Minutes format differs from TraditionalV3!
# =============================================================================
def parse_minutes_advanced_v3(minutes_str: str) -> float:
    """
    AdvancedV3 uses 'MM:SS' format (e.g., '28:17')
    TraditionalV3 uses 'PT28M17.00S' format

    Handle both for safety.
    """
    if pd.isna(minutes_str) or minutes_str == "" or minutes_str is None:
        return 0.0

    minutes_str = str(minutes_str)

    # Try MM:SS format first (AdvancedV3)
    if ":" in minutes_str and "PT" not in minutes_str:
        try:
            parts = minutes_str.split(":")
            mins = int(parts[0])
            secs = int(parts[1]) if len(parts) > 1 else 0
            return mins + secs / 60
        except:
            pass

    # Try ISO 8601 format (TraditionalV3)
    if "PT" in minutes_str:
        try:
            import re

            match = re.match(r"PT(\d+)M([\d.]+)S", minutes_str)
            if match:
                mins = int(match.group(1))
                secs = float(match.group(2))
                return mins + secs / 60
        except:
            pass

    return 0.0


# =============================================================================
# ESTIMATED vs ACTUAL - WHICH TO USE?
# =============================================================================
ESTIMATED_VS_ACTUAL = """
NBA provides both 'estimated' and actual versions of some metrics.
    
Differences:
- Actual: Calculated directly from box score events
- Estimated: Uses statistical models to account for lineup context

Recommendation:
- For TRAINING: Use actual (offensiveRating, defensiveRating, usagePercentage)
- The estimated versions add noise and are less interpretable

Exception:
- If you're doing lineup-adjusted analysis, estimated may be more appropriate
- But for standard player evaluation, stick with actual
"""

# =============================================================================
# COLUMNS TO SCRAPE (FINAL LIST)
# =============================================================================
ADVANCED_V3_SCRAPE_COLUMNS = [
    # Identifiers
    "gameId",
    "teamId",
    "personId",
    # Time
    "minutes",
    # Core ratings (USE THESE)
    "offensiveRating",
    "defensiveRating",
    "netRating",
    # Core percentages (USE THESE)
    "usagePercentage",
    "trueShootingPercentage",
    "effectiveFieldGoalPercentage",
    "assistPercentage",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    # Other useful
    "turnoverRatio",
    "pace",
    "possessions",
    "PIE",
]

# Skip these (redundant or less useful)
ADVANCED_V3_SKIP_COLUMNS = [
    "estimatedOffensiveRating",  # Use actual
    "estimatedDefensiveRating",  # Use actual
    "estimatedNetRating",  # Use actual
    "estimatedUsagePercentage",  # Use actual
    "estimatedPace",  # Use actual
    "pacePer40",  # Redundant with pace
    "assistToTurnover",  # Can compute from Traditional
    "assistRatio",  # assistPercentage is more standard
    "reboundPercentage",  # Have ORB% and DRB% separately
]

print("BoxScoreAdvancedV3 Schema Reference loaded.")
print(f"  - PlayerStats: {sum(len(v) for v in PLAYER_ADVANCED_V3.values() if isinstance(v, dict))} fields")
print(f"  - Recommended scrape columns: {len(ADVANCED_V3_SCRAPE_COLUMNS)}")
print(f"  - Skip columns: {len(ADVANCED_V3_SKIP_COLUMNS)}")

BoxScoreAdvancedV3 Schema Reference loaded.
  - PlayerStats: 30 fields
  - Recommended scrape columns: 17
  - Skip columns: 9


In [12]:
##3.3 BoxScoreAdvancedV2 (Team level) - Per-game team metrics

In [15]:
# We already loaded this above, but let's look at team-level data (usually index 1)
box_adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=TEST_GAME_ID)
rate_limit()

dfs = box_adv.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")

# Index 1 is usually team stats
if len(dfs) > 1:
    team_box = dfs[1]
    print(f"\nTeam Box Score Shape: {team_box.shape}")
    print(f"Columns: {list(team_box.columns)}")
    display(team_box)
dfs

Number of dataframes: 2

Team Box Score Shape: (2, 30)
Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'estimatedTeamTurnoverPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,minutes,estimatedOffensiveRating,offensiveRating,estimatedDefensiveRating,defensiveRating,estimatedNetRating,netRating,assistPercentage,assistToTurnover,assistRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,estimatedTeamTurnoverPercentage,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,0022300001,1610612754,Indiana,Pacers,IND,pacers,240:00,593.1,118.6,563.1,112.6,30.0,6.0,0.622,1.47,19.6,0.318,0.822,0.573,18.627,18.6,0.610,0.627,1.0,0.991,20.5,102.5,85.42,102.0,0.493
1,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,240:00,563.1,112.6,593.1,118.6,-30.0,-6.0,0.614,2.08,19.6,0.178,0.682,0.427,12.621,12.6,0.571,0.611,1.0,0.973,20.5,102.5,85.42,103.0,0.507


[        gameId      teamId   teamCity   teamName teamTricode   teamSlug  \
 0   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 1   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 2   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 3   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 4   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 5   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 6   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 7   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 8   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 9   0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 10  0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 11  0022300001  1610612739  Cleveland  Cavaliers         CLE  cavaliers   
 12  0022300

In [ ]:
TEAM_ADVANCED_V3 = {
    "identifiers": {
        "gameId": "Game identifier",
        "teamId": "Team identifier",
        "teamTricode": "Team abbreviation (e.g., CLE, IND)",
    },
    "ratings": {
        "offensiveRating": "Points scored per 100 possessions",
        "defensiveRating": "Points allowed per 100 possessions",
        "netRating": "ORTG - DRTG",
        # Skip estimated versions
    },
    "percentages": {
        "effectiveFieldGoalPercentage": "eFG%",
        "trueShootingPercentage": "TS%",
        "assistPercentage": "AST%",
        "offensiveReboundPercentage": "ORB%",
        "defensiveReboundPercentage": "DRB%",
        "turnoverRatio": "TOV per 100 possessions",
    },
    "pace": {
        "pace": "Possessions per 48 minutes",
        "possessions": "Total possessions in game",
    },
    "other": {
        "PIE": "Player Impact Estimate (team aggregate)",
    },
}

TEAM_ADVANCED_V3_SCRAPE_COLUMNS = [
    "gameId",
    "teamId",
    "teamTricode",
    "offensiveRating",
    "defensiveRating",
    "netRating",
    "effectiveFieldGoalPercentage",
    "trueShootingPercentage",
    "assistPercentage",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "turnoverRatio",
    "pace",
    "possessions",
]

print("BoxScoreAdvancedV3 Team Schema loaded.")
print(f"  - Recommended columns: {len(TEAM_ADVANCED_V3_SCRAPE_COLUMNS)}")